# Module 21 — Week 10 — Bayesian Black-Box Optimisation Capstone

**W9: 3/8 improved (F1 ⭐, F5 ⭐, F7 ⭐ — all new bests). F8 missed by 0.0005. Fix: F2 recovery to W6 exact, F3 x3 corrected toward W1, F6 recovery to W8, F8 ultra-tight.**

In [ ]:
# ── Cell 1: Load all the tools (libraries) we need ──────────────────────────

import matplotlib               # for drawing graphs
matplotlib.use('Agg')           # save graphs to file — no popup on screen

import numpy as np              # for all number and array operations
import matplotlib.pyplot as plt # for creating charts

# GaussianProcessRegressor = our smart model that learns from past data
# Matern = tells the GP "assume the function changes smoothly"
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern

# norm = normal distribution math (used inside EI formula)
# yeojohnson = a special transform for F4's wild output values
from scipy.stats import norm, yeojohnson

# LatinHypercube = smart way to spread 50,000 test points evenly across space
from scipy.stats.qmc import LatinHypercube

import warnings                   # for managing warning messages
import os                         # for file and folder operations

warnings.filterwarnings('ignore') # hide warnings so output stays clean

# Folder where we save our progress graphs
PLOTS_DIR = '/Users/luckydhanvi/Documents/DataScience/PracticalLive/LEVEL_2/imperial-aiml-capstone/module-21/plots'
os.makedirs(PLOTS_DIR, exist_ok=True)  # create folder if it does not exist yet

print('All libraries loaded — Week 10 ready!')

In [ ]:
submitted_x_w1={1:[0.020584,0.969910],2:[0.814691,0.969505],3:[0.376075,0.370839,0.474761],4:[0.369789,0.452786,0.367951,0.448446],5:[0.241041,0.805036,0.948951,0.905090],6:[0.466959,0.356875,0.489683,0.726384,0.125125],7:[0.027698,0.531762,0.337094,0.176133,0.361503,0.730849],8:[0.192432,0.183093,0.018724,0.036362,0.690267,0.444236,0.081374,0.428967]}
new_y_w1={1:1.966e-321,2:0.1292261555216582,3:-0.010707313301147062,4:-0.34595283782499875,5:1450.9433021815964,6:-0.3611823990070205,7:1.4058168801082682,8:9.8915570907296}
submitted_x_w2={1:[0.591837,0.591837],2:[0.000000,1.000000],3:[0.421053,1.000000,1.000000],4:[0.909548,0.568955,0.762175,0.811807],5:[0.204881,0.877830,0.879582,0.870578],6:[0.851439,0.906254,0.506372,0.594105,0.708147],7:[0.097054,0.432660,0.338116,0.122619,0.296117,0.886436],8:[0.076274,0.101214,0.383035,0.338493,0.113685,0.882235,0.615428,0.796463]}
new_y_w2={1:0.00028209052469858225,2:0.1709619176069506,3:-0.48304244384724265,4:-26.59459580774249,5:1192.2995655092311,6:-1.9259411859252866,7:1.2030170341293975,8:9.0382459830856}
submitted_x_w3={1:[0.980000,0.980000],2:[1.000000,0.306122],3:[1.000000,0.000000,0.684211],4:[0.985601,0.686679,0.243615,0.798556],5:[0.204881,0.877830,0.879582,0.870578],6:[0.061416,0.762464,0.106527,0.271402,0.782742],7:[0.067189,0.412831,0.295130,0.070570,0.412599,0.616173],8:[0.682757,0.427203,0.591529,0.734064,0.514947,0.813984,0.722156,0.615073]}
new_y_w3={1:2.665897212344236e-174,2:-0.042550557700427774,3:-0.1840890683677661,4:-26.07041694623693,5:1192.2995655092311,6:-2.508952125110497,7:1.2533263563752521,8:7.5792591902086}
submitted_x_w4={1:[0.278296,0.020000],2:[0.685269,0.947006],3:[0.403468,0.441923,0.497061],4:[0.352971,0.651614,0.805417,0.616108],5:[0.167299,0.881015,0.978872,0.954244],6:[0.334649,0.293944,0.500782,0.769829,0.074923],7:[0.206363,0.281987,0.389442,0.281544,0.218827,0.711599],8:[0.095545,0.327238,0.051339,0.269531,0.555763,0.417489,0.285113,0.613881]}
new_y_w4={1:-2.6647756688938686e-133,2:0.14705786268424045,3:-0.022992940111015336,4:-0.1283640964538999,5:2496.347187728138,6:-0.38592078647528016,7:2.6705394912160187,8:9.8967959631939}
submitted_x_w5={1:[0.580092,0.683225],2:[0.702813,0.926626],3:[0.365086,0.316421,0.471038],4:[0.344700,0.645505,0.791987,0.622463],5:[0.139557,0.911522,0.979905,0.977049],6:[0.417831,0.356959,0.468069,0.668531,0.039515],7:[0.384097,0.122113,0.444891,0.357064,0.147383,0.783086],8:[0.089787,0.068251,0.180968,0.327284,0.766207,0.653365,0.174832,0.499246]}
new_y_w5={1:0.00008112997850454906,2:0.5833602539566602,3:-0.018707796769607724,4:-13.979947691578896,5:2941.854350298978,6:-0.29985775692426564,7:1.660798293687705,8:9.9275118625839}
submitted_x_w6={1:[0.653384,0.652924],2:[0.704856,0.921380],3:[0.151659,0.826046,0.622768],4:[0.291709,0.714786,0.911879,0.664486],5:[0.102387,0.951309,0.977662,0.978193],6:[0.394360,0.399099,0.411100,0.595076,0.020599],7:[0.027785,0.208441,0.270801,0.266138,0.195016,0.698660],8:[0.029320,0.285338,0.223021,0.041430,0.625367,0.719031,0.033888,0.797446]}
new_y_w6={1:0.0880341468685302,2:0.6478060146282238,3:-0.0867832750698683,4:-21.252967893168208,5:3303.918327634855,6:-0.5181323257010382,7:2.1498244177996053,8:9.8116383300479}
submitted_x_w7={1:[0.533625,0.533688],2:[0.702665,0.919352],3:[0.267218,0.472728,0.513126],4:[0.341374,0.485949,0.606273,0.478470],5:[0.071951,0.977037,0.979767,0.979593],6:[0.441360,0.279908,0.514886,0.684556,0.020780],7:[0.255243,0.272333,0.253655,0.238536,0.237243,0.658362],8:[0.306263,0.307104,0.109816,0.361473,0.669399,0.417361,0.175248,0.274400]}
new_y_w7={1:3.8800114757386434e-13,2:0.6475701405048025,3:-0.038391302132802174,4:-4.940966055787715,5:3626.8315773030813,6:-0.33902767001927475,7:2.5915976846312665,8:9.8171020976195}
submitted_x_w8={1:[0.643797,0.704343],2:[0.703797,0.924514],3:[0.100454,0.240875,0.185908],4:[0.383806,0.511512,0.656499,0.491038],5:[0.044074,0.979912,0.978237,0.978721],6:[0.409346,0.360704,0.502905,0.718263,0.020264],7:[0.138618,0.329866,0.325614,0.264255,0.295279,0.651123],8:[0.182189,0.037358,0.268170,0.170921,0.585250,0.468337,0.247759,0.632930]}
new_y_w8={1:-0.00011412865302137135,2:0.6409162660536529,3:-0.14148000083888568,4:-7.1627894437582,5:3632.183153202294,6:-0.2037089488415273,7:2.7440435657471656,8:9.887359571982}
submitted_x_w9={1:[0.651500,0.654000],2:[0.706000,0.921000],3:[0.020793,0.975360,0.381751],4:[0.354000,0.650000,0.806000,0.617000],5:[0.027000,0.980000,0.979500,0.979000],6:[0.420721,0.410509,0.539518,0.760163,0.022635],7:[0.194674,0.230232,0.307303,0.258261,0.265579,0.680946],8:[0.090500,0.066000,0.182000,0.330000,0.768000,0.650000,0.175000,0.502000]}
new_y_w9={1:0.09715493565789202,2:0.5975456023703872,3:-0.05325899029920575,4:-14.466436970145782,5:3651.372623492115,6:-0.28461851247025494,7:2.9422351452635813,8:9.9270291}
all_time_best={1:(0.09715493565789202,'W9'),2:(0.6478060146282238,'W6'),3:(-0.010707313301147062,'W1'),4:(-0.1283640964538999,'W4'),5:(3651.372623492115,'W9'),6:(-0.2037089488415273,'W8'),7:(2.9422351452635813,'W9'),8:(9.9275118625839,'W5')}
print('Historical data loaded W1-W9')
for i in range(1,9):
    v,w=all_time_best[i]; print(f'  F{i}: {v:.4e} ({w})')

In [ ]:
# ── Cell 3: Load initial data files + add all 9 weekly submissions ───────────
# The .npy files contain 10 starting points given to us at the start of the project
# We stack our weekly submissions on top to build the full dataset for each function

BASE_PATH = '/Users/luckydhanvi/Documents/DataScience/PracticalLive/LEVEL_2/imperial-aiml-capstone/data/'

descriptions = {
    1: 'Radiation Detection',   2: 'Noisy ML Model',
    3: 'Drug Discovery',        4: 'Warehouse Placement',
    5: 'Chemical Yield (STAR)', 6: 'Cake Recipe',
    7: 'ML Hyperparameters',    8: 'Complex 8D'
}

data = {}  # will hold X (inputs) and Y (outputs) for each function

for i in range(1, 9):

    # Load the 10 initial data points from file
    X_initial = np.load(f'{BASE_PATH}function_{i}/initial_inputs.npy')
    Y_initial = np.load(f'{BASE_PATH}function_{i}/initial_outputs.npy')

    # Stack all 9 weekly submissions below the initial data
    # reshape(1,-1) turns a list [0.65, 0.65] into a row [[0.65, 0.65]]
    X_all = np.vstack([
        X_initial,
        np.array(submitted_x_w1[i]).reshape(1, -1),
        np.array(submitted_x_w2[i]).reshape(1, -1),
        np.array(submitted_x_w3[i]).reshape(1, -1),
        np.array(submitted_x_w4[i]).reshape(1, -1),
        np.array(submitted_x_w5[i]).reshape(1, -1),
        np.array(submitted_x_w6[i]).reshape(1, -1),
        np.array(submitted_x_w7[i]).reshape(1, -1),
        np.array(submitted_x_w8[i]).reshape(1, -1),
        np.array(submitted_x_w9[i]).reshape(1, -1),
    ])

    # Join all 9 weekly results with the initial outputs into one list
    Y_all = np.concatenate([
        Y_initial,
        [new_y_w1[i]], [new_y_w2[i]], [new_y_w3[i]],
        [new_y_w4[i]], [new_y_w5[i]], [new_y_w6[i]],
        [new_y_w7[i]], [new_y_w8[i]], [new_y_w9[i]],
    ])

    data[i] = {'X': X_all, 'Y': Y_all}

# Print a summary of what we loaded
print('Data loaded for all 8 functions!')
print()
for i in range(1, 9):
    n      = len(data[i]['Y'])
    best_v = all_time_best[i][0]
    best_w = all_time_best[i][1]
    print(f'  F{i} ({descriptions[i]}):  {n} points  |  all-time best = {best_v:.4f} ({best_w})')

In [ ]:
# ── Cell 4: Helper functions ─────────────────────────────────────────────────

# We never search at the very edges (0 or 1) — corners give bad results
LOWER_BOUND = 0.02
UPPER_BOUND = 0.98


# ─── Helper A: Transform outputs to a nicer scale for the GP ─────────────────
# GP works better when all output values are on a similar scale
# Example: F5 outputs ~3651, F3 outputs ~-0.01  (very different!)
# After log-transform both become small numbers the GP can handle easily
def transform_outputs(Y, method='log'):
    if method == 'log':
        # log of absolute value, keep the original sign
        return np.log(np.abs(Y) + 1e-300) * np.sign(Y + 1e-300)
    elif method == 'yeojohnson':
        # Yeo-Johnson handles F4's extremely wild output values better
        Y_transformed, _ = yeojohnson(Y)
        return Y_transformed
    else:
        return Y.copy()  # no transform needed


# ─── Helper B: Calculate Expected Improvement (EI) score ─────────────────────
# EI answers: "how much better than our current best do we expect this point to be?"
# Points with high EI are worth trying — they might improve our best result
def expected_improvement(mu, sigma, current_best, xi=0.01):
    improvement = mu - current_best - xi   # how much better than current best
    Z  = improvement / (sigma + 1e-9)      # normalise (avoid divide-by-zero)
    ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
    ei[sigma < 1e-10] = 0.0               # zero EI where GP is already certain
    return ei


# ─── Helper C: The main analysis function ────────────────────────────────────
# Call this once for each function to get the Week 10 portal submission string
def analyse_w10(func_num, beta, trust_center, trust_radius,
                xi=0.01, alpha=1e-6, y_transform='log', policy='explore'):

    # ── Step 1: Get all our collected data for this function ──────────────────
    X   = data[func_num]['X']   # every input point we have tried so far
    Y   = data[func_num]['Y']   # every output result from the portal
    dim = X.shape[1]            # how many dimensions (2, 3, 4, 5, 6, or 8)

    # ── Step 2: Find the best result we have ever seen ────────────────────────
    best_idx = np.argmax(Y)     # position of the highest output in Y
    best_Y   = Y[best_idx]      # the highest output value itself

    # Print a header so we know which function we are analysing
    print(f'\n{"="*65}')
    print(f'F{func_num} — {descriptions[func_num]} ({dim}D)')
    print(f'Policy      : {policy.upper()}')
    print(f'Data points : {len(Y)}')
    print(f'Best output ever  : {best_Y:.6f}')
    print(f'Week 9 output was : {new_y_w9[func_num]:.6f}')
    print(f'Trust center : {np.round(trust_center, 4).tolist()}')
    print(f'Trust radius : {trust_radius}   |   Beta : {beta}')
    print('='*65)

    # ── Step 3: Print a sorted table of our top 5 observations ───────────────
    sorted_pairs = sorted(zip(Y, range(len(Y))), reverse=True)
    print(f'\n  Top 5 observations (best to worst):')
    print(f'  {"-"*55}')
    for rank, (y_val, idx) in enumerate(sorted_pairs[:5]):
        marker = '  ← BEST' if rank == 0 else ''
        x_str  = ', '.join([f'{v:.4f}' for v in X[idx]])
        print(f'  [{rank+1}]  Y = {y_val:+.4f}   X = [{x_str}]{marker}')
    print(f'  {"-"*55}')

    # ── Step 4: Transform outputs so GP can learn better ─────────────────────
    Y_transformed = transform_outputs(Y, method=y_transform)
    print(f'\n  Output transform : {y_transform}')

    # ── Step 5: Generate 50,000 candidate points inside the trust region ──────
    # Trust region = a small box around the best known point
    # We only search inside this box — no point wandering far away at week 10
    box_lo = np.clip(trust_center - trust_radius, LOWER_BOUND, UPPER_BOUND)
    box_hi = np.clip(trust_center + trust_radius, LOWER_BOUND, UPPER_BOUND)

    sampler     = LatinHypercube(d=dim, seed=42)   # evenly spread, not random clumps
    raw_samples = sampler.random(n=50000)
    X_candidates = box_lo + raw_samples * (box_hi - box_lo)  # scale into our box

    print(f'  Generated 50,000 candidate points inside trust region')
    print(f'  Box: {np.round(box_lo,3).tolist()}  →  {np.round(box_hi,3).tolist()}')

    # ── Step 6: Train the Gaussian Process on all our past data ──────────────
    # The GP reads our (X, Y) history and builds a prediction surface
    # It can then tell us what output ANY untried point might give
    kernel = Matern(length_scale=0.2, nu=2.5)  # Matern = "assume smooth function"
    gp = GaussianProcessRegressor(
        kernel             = kernel,
        alpha              = alpha,       # small noise tolerance for stability
        n_restarts_optimizer = 3,         # try 3 starting points for best GP fit
        normalize_y        = True         # internally rescale Y for better fitting
    )
    gp.fit(X, Y_transformed)
    print(f'  GP trained on {len(Y)} data points')

    # ── Step 7: Ask the GP to score all 50,000 candidates ────────────────────
    mu, sigma = gp.predict(X_candidates, return_std=True)
    # mu    = GP predicted output at each candidate (what it thinks the portal will return)
    # sigma = GP uncertainty at each candidate (high = "I am not sure here")

    # ── Step 8: Calculate UCB score for every candidate ───────────────────────
    # UCB = predicted output + (beta × uncertainty)
    # Low beta  (0.02) → trust prediction, exploit known good area
    # High beta (0.8+) → chase uncertainty, explore unknown areas
    ucb_scores     = mu + beta * sigma
    best_ucb_idx   = np.argmax(ucb_scores)
    best_ucb_point = X_candidates[best_ucb_idx]
    print(f'\n  UCB winner score : {ucb_scores[best_ucb_idx]:.4f}')

    # ── Step 9: Calculate EI score for every candidate ───────────────────────
    # EI = "expected improvement over current best"
    current_best_transformed = float(
        transform_outputs(np.array([best_Y]), method=y_transform)[0]
    )
    ei_scores     = expected_improvement(mu, sigma, current_best_transformed, xi=xi)
    best_ei_idx   = np.argmax(ei_scores)
    best_ei_point = X_candidates[best_ei_idx]
    print(f'  EI  winner score : {ei_scores[best_ei_idx]:.6f}')

    # ── Step 10: Pick the winner — UCB or EI? ────────────────────────────────
    # Ask the GP to predict both winning points
    # Whichever has the higher predicted output wins
    mu_ucb = float(gp.predict(best_ucb_point.reshape(1, -1)).ravel()[0])
    mu_ei  = float(gp.predict(best_ei_point.reshape(1, -1)).ravel()[0])

    if mu_ei >= mu_ucb:
        next_x = best_ei_point
        winner = 'EI'
    else:
        next_x = best_ucb_point
        winner = 'UCB'
    print(f'  Winner : {winner}  (GP mean UCB={mu_ucb:.4f}, EI={mu_ei:.4f})')

    # ── Step 11: Duplicate check ──────────────────────────────────────────────
    # If the new point is too close (< 0.015) to something we already submitted,
    # add a tiny random nudge so we do not waste the query on the same spot again
    prior_submissions = [
        np.array(submitted_x_w1[func_num]), np.array(submitted_x_w2[func_num]),
        np.array(submitted_x_w3[func_num]), np.array(submitted_x_w4[func_num]),
        np.array(submitted_x_w5[func_num]), np.array(submitted_x_w6[func_num]),
        np.array(submitted_x_w7[func_num]), np.array(submitted_x_w8[func_num]),
        np.array(submitted_x_w9[func_num]),
    ]
    min_dist = min(np.linalg.norm(next_x - p) for p in prior_submissions)
    print(f'  Distance from nearest prior submission : {min_dist:.4f}')

    if min_dist < 0.015:
        np.random.seed(99)
        noise  = np.random.uniform(-0.02, 0.02, dim)   # small random nudge
        next_x = np.clip(next_x + noise, LOWER_BOUND, UPPER_BOUND)
        print(f'  ⚠ Too close — added small noise to avoid duplicate')

    # ── Step 12: Format as portal submission string and print ─────────────────
    portal_string    = '-'.join([f'{v:.6f}' for v in next_x])
    dist_from_center = np.linalg.norm(next_x - trust_center)
    print(f'  Distance from trust center : {dist_from_center:.4f}')
    print(f'\n  >>> SUBMIT F{func_num}: {portal_string} <<<')

    return next_x, portal_string


print('Helper functions ready!')

---
## F1 — Radiation Detection (2D)
**W9 result: 0.09715 ✅ NEW BEST**

| Setting | Value | Reason |
|---------|-------|--------|
| Policy | MOMENTUM | Got new best last week — keep pushing |
| Trust center | W9 best (0.6515, 0.6540) | Best result we ever had |
| Trust radius | 0.02 | Very tight box — micro-exploit |
| Beta | 0.3 | Low — trust GP prediction, not uncertainty |
| Transform | log | Standard |

In [ ]:
# W9 best point — our confirmed all-time best for F1
W9_BEST_F1 = np.array([0.651500, 0.654000])

# Run the GP analysis for F1
next_x1, portal1 = analyse_w10(
    func_num     = 1,
    beta         = 0.3,          # low beta = exploit, not explore
    trust_center = W9_BEST_F1,   # search around W9 best
    trust_radius = 0.02,         # very tight box (r=0.02 in each direction)
    xi           = 0.001,        # small EI margin
    y_transform  = 'log',        # standard log transform
    policy       = 'momentum'
)

# ── Manual override ───────────────────────────────────────────────────────────
# Problem: GP drifted x2 to 0.674, but our confirmed best has x2 = 0.654
# Reason : F1 may have a very narrow spike the GP kernel averages away
# Fix    : Manually correct x2 back toward 0.654 (near confirmed W9 best)
portal1 = '0.651000-0.654500'
next_x1 = np.array([0.651000, 0.654500])
print()
print('  MANUAL OVERRIDE applied:')
print('  GP suggested x2=0.674 but W9 best was at x2=0.654')
print(f'  Final submission: {portal1}')

---
## F2 — Noisy ML Model (2D)
**W9 result: 0.5975 ❌ Dropped from 0.6478**

| Setting | Value | Reason |
|---------|-------|--------|
| Policy | RECOVERY | W9 moved x1 by +0.001 and dropped by 0.05 — very sensitive |
| Trust center | W6 best (0.7049, 0.9214) | Best result we ever had |
| Trust radius | 0.015 | Tight — return to safe confirmed best |
| Beta | 0.2 | Very low — exploit only |
| Transform | log | Standard |

In [ ]:
# W6 best point — confirmed all-time best for F2
W6_BEST_F2 = np.array([0.704856, 0.921380])

# Run the GP analysis for F2
next_x2, portal2 = analyse_w10(
    func_num     = 2,
    beta         = 0.2,          # very low — exploit only
    trust_center = W6_BEST_F2,   # return to W6 confirmed best
    trust_radius = 0.015,        # tight — F2 is very sensitive to small x1 changes
    xi           = 0.005,
    y_transform  = 'log',
    policy       = 'recovery'
)

---
## F3 — Drug Discovery (3D)
**W9 result: -0.0533 ❌ Stuck since Week 1 (-0.0107)**

| Setting | Value | Reason |
|---------|-------|--------|
| Policy | ANCHOR-TIGHT | W9 used x3=0.382, but W1 best used x3=0.475 — fix x3 |
| Trust center | W1 best (0.0206, 0.9699, 0.4748) | Only best we ever had |
| Trust radius | 0.06 | Moderate — stay near W1 but let GP fix x3 |
| Beta | 0.8 | Slightly higher — allow some exploration |
| Transform | log | Standard |

In [ ]:
# W1 best point — still the all-time best for F3 after 9 weeks!
W1_BEST_F3 = np.array([0.020584, 0.969910, 0.474761])

# Run the GP analysis for F3
# Key insight: W9 used x3=0.382, but W1 best used x3=0.475
# Trust radius 0.06 forces the GP to stay near x3=0.475
next_x3, portal3 = analyse_w10(
    func_num     = 3,
    beta         = 0.8,          # slightly higher — allow GP to find correct x3
    trust_center = W1_BEST_F3,   # anchor to W1 best
    trust_radius = 0.06,         # moderate — enough room to correct x3
    xi           = 0.001,
    y_transform  = 'log',
    policy       = 'anchor-tight'
)

---
## F4 — Warehouse Placement (4D)
**W9 result: -14.466 ❌ Near-exact W4 coords gave terrible result**

| Setting | Value | Reason |
|---------|-------|--------|
| Policy | RECENT-ONLY DYNAMIC | F4 landscape may change over time — use only recent data |
| Trust center | W4 best (0.3530, 0.6516, 0.8054, 0.6161) | Best we ever had |
| Trust radius | 0.05 | Slightly wider — recent-only GP needs a bit more room |
| Beta | 0.5 | Balanced |
| Transform | yeojohnson | F4 outputs are very wild — Yeo-Johnson handles them better |

**Special setup:** We use only recent observations (initial data + W4, W7, W8, W9).
We skip W2/W3 (outliers: -26.59, -26.07) and W5/W6 (possibly stale landscape).

In [ ]:
# ── Special setup for F4: use only recent observations ───────────────────────
# Hypothesis: F4 may be a dynamic function (landscape changes over time)
# Evidence  : Near-exact W4 coords gave -14.466 in W9, but -0.128 in W4
# Fix       : Only use recent data (initial + W4, W7, W8, W9)
#             Skip W2/W3 (outliers) and W5/W6 (possibly stale landscape)

# Load the original initial data file
init_X4 = np.load('/Users/luckydhanvi/Documents/DataScience/PracticalLive/LEVEL_2/imperial-aiml-capstone/data/function_4/initial_inputs.npy')
init_Y4 = np.load('/Users/luckydhanvi/Documents/DataScience/PracticalLive/LEVEL_2/imperial-aiml-capstone/data/function_4/initial_outputs.npy')

# Remove extreme outliers from the initial data (keep only outputs > -20)
clean_mask = init_Y4 > -20
init_X4_clean = init_X4[clean_mask]
init_Y4_clean = init_Y4[clean_mask]

# Our 4 selected weekly observations (recent only)
recent_X4 = np.array([
    [0.352971, 0.651614, 0.805417, 0.616108],   # W4 result: -0.1284 (ALL-TIME BEST)
    [0.341374, 0.485949, 0.606273, 0.478470],   # W7 result: -4.94
    [0.383806, 0.511512, 0.656499, 0.491038],   # W8 result: -7.163
    [0.354000, 0.650000, 0.806000, 0.617000],   # W9 result: -14.466
])
recent_Y4 = np.array([-0.1284, -4.94, -7.163, -14.466])

# Combine clean initial data + recent weekly submissions
data[4]['X'] = np.vstack([init_X4_clean, recent_X4])
data[4]['Y'] = np.concatenate([init_Y4_clean, recent_Y4])

print(f'F4 recent-only dataset ready: {len(data[4]["Y"])} points total')
print(f'(Initial clean: {len(init_Y4_clean)}  +  Recent weekly: {len(recent_Y4)})')

# W4 best point — best we ever had for F4
W4_BEST_F4 = np.array([0.352971, 0.651614, 0.805417, 0.616108])

# Run the GP analysis for F4 with recent-only data
next_x4, portal4 = analyse_w10(
    func_num     = 4,
    beta         = 0.5,           # balanced
    trust_center = W4_BEST_F4,    # still anchor to all-time best
    trust_radius = 0.05,          # slightly wider (recent-only GP needs more room)
    xi           = 0.01,
    alpha        = 0.1,           # higher alpha = more tolerance for F4's noise
    y_transform  = 'yeojohnson',  # Yeo-Johnson handles F4's wild values better
    policy       = 'recent-only-dynamic'
)

---
## F5 — Chemical Yield STAR (4D)
**W9 result: 3651.37 ✅ NEW BEST — best in class!**

| Setting | Value | Reason |
|---------|-------|--------|
| Policy | MOMENTUM | Upward trend every week — keep riding the ridge |
| Trust center | W9 best (0.027, 0.980, 0.9795, 0.979) | Best ever |
| Trust radius | 0.015 | Very tight — ridge is narrow |
| Beta | 0.02 | Tiny — fully exploit, no exploration |
| Transform | log | Standard |

**Ridge pattern in x1:** 0.241 → 0.140 → 0.102 → 0.072 → 0.044 → 0.027 → **0.010**
x2, x3, x4 stay at 0.979–0.980 every week.

In [ ]:
# W9 best point — confirmed all-time best for F5
W9_BEST_F5 = np.array([0.027000, 0.980000, 0.979500, 0.979000])

# Run the GP analysis for F5
next_x5, portal5 = analyse_w10(
    func_num     = 5,
    beta         = 0.02,         # tiny beta — fully exploit, zero exploration
    trust_center = W9_BEST_F5,   # search around W9 best
    trust_radius = 0.015,        # very tight — the ridge is narrow
    xi           = 0.01,
    y_transform  = 'log',
    policy       = 'momentum'
)

# ── Manual override ───────────────────────────────────────────────────────────
# Problem: GP sometimes perturbs x1 backwards due to the duplicate check
# Reason : There is a clear downward ridge in x1 every single week
# Ridge  : 0.241 → 0.140 → 0.102 → 0.072 → 0.044 → 0.027 → 0.010
# Fix    : Manually set x1=0.010, hold x2-x4 at confirmed ridge values
portal5 = '0.010000-0.980000-0.979500-0.979000'
next_x5 = np.array([0.010000, 0.980000, 0.979500, 0.979000])
print()
print('  MANUAL OVERRIDE applied:')
print('  Ridge trend: 0.241→0.140→0.102→0.072→0.044→0.027→0.010')
print(f'  Final submission: {portal5}')

---
## F6 — Cake Recipe (5D)
**W9 result: -0.2846 ❌ Dropped from W8 best -0.2037**

| Setting | Value | Reason |
|---------|-------|--------|
| Policy | RECOVERY | W9 pushed x4 too high (0.760) — return to W8 best |
| Trust center | W8 best (0.4093, 0.3607, 0.5029, 0.7183, 0.0203) | Best ever |
| Trust radius | 0.03 | Tight — return to confirmed W8 region |
| Beta | 0.2 | Low — exploit only |
| Transform | log | Standard |

In [ ]:
# W8 best point — confirmed all-time best for F6
W8_BEST_F6 = np.array([0.409346, 0.360704, 0.502905, 0.718263, 0.020264])

# Run the GP analysis for F6
next_x6, portal6 = analyse_w10(
    func_num     = 6,
    beta         = 0.2,          # low — exploit only
    trust_center = W8_BEST_F6,   # return to W8 confirmed best
    trust_radius = 0.03,         # tight — W9 went wrong by pushing x4 too high
    xi           = 0.001,
    y_transform  = 'log',
    policy       = 'recovery'
)

---
## F7 — ML Hyperparameters (6D)
**W9 result: 2.9422 ✅ NEW BEST — 4 consecutive improvements!**

| Setting | Value | Reason |
|---------|-------|--------|
| Policy | MOMENTUM | Trend: 2.150 → 2.592 → 2.744 → 2.942 — keep going |
| Trust center | W9 best (0.1947, 0.2302, 0.3073, 0.2583, 0.2656, 0.6809) | Best ever |
| Trust radius | 0.08 | Moderate — 6D needs a bit more room than 2D/4D |
| Beta | 0.5 | Balanced — slight exploration |
| Transform | log | Standard |

In [ ]:
# W9 best point — confirmed all-time best for F7
W9_BEST_F7 = np.array([0.194674, 0.230232, 0.307303, 0.258261, 0.265579, 0.680946])

# Run the GP analysis for F7
next_x7, portal7 = analyse_w10(
    func_num     = 7,
    beta         = 0.5,          # balanced — allow slight exploration in 6D space
    trust_center = W9_BEST_F7,   # continue from W9 best
    trust_radius = 0.08,         # moderate — 6D needs more room than 2D
    xi           = 0.01,
    y_transform  = 'log',
    policy       = 'momentum'
)

---
## F8 — Complex 8D
**W9 result: 9.9270 ❌ Missed W5 best (9.9275) by only 0.0005!**

| Setting | Value | Reason |
|---------|-------|--------|
| Policy | ULTRA-TIGHT RECOVERY | So close to best — tightest radius ever tried |
| Trust center | W5 best (0.0898, 0.0683, 0.1810, 0.3273, 0.7662, 0.6534, 0.1748, 0.4992) | Best ever |
| Trust radius | 0.04 | Very tight — 8D is hard, stay close to confirmed best |
| Beta | 0.3 | Low — exploit |
| Transform | log | Standard |

In [ ]:
# W5 best point — confirmed all-time best for F8 (9.9275)
W5_BEST_F8 = np.array([0.089787, 0.068251, 0.180968, 0.327284, 0.766207, 0.653365, 0.174832, 0.499246])

# Run the GP analysis for F8
# W9 result was 9.9270 — missed by only 0.0005
# Strategy: go even tighter to find those last 0.0005 points
next_x8, portal8 = analyse_w10(
    func_num     = 8,
    beta         = 0.3,          # low — exploit
    trust_center = W5_BEST_F8,   # anchor to W5 confirmed best
    trust_radius = 0.04,         # tightest we have ever used for F8
    xi           = 0.01,
    y_transform  = 'log',
    policy       = 'ultra-tight-recovery'
)

In [ ]:
# ── Final Summary: Print all 8 portal submission strings ─────────────────────

print('=' * 70)
print('WEEK 10 — MODULE 21 — PORTAL SUBMISSION STRINGS')
print('=' * 70)
print()

# Collect all portal strings into one place
all_portals = {
    1: portal1,
    2: portal2,
    3: portal3,
    4: portal4,
    5: portal5,
    6: portal6,
    7: portal7,
    8: portal8,
}

# Policy label for each function (for display)
all_policies = {
    1: 'MOMENTUM        (manual — x2 corrected back to 0.654)',
    2: 'RECOVERY        (return to W6 best, r=0.015)',
    3: 'ANCHOR-TIGHT    (W1 best, x3 corrected toward 0.475)',
    4: 'RECENT-ONLY     (W4+W7+W8+W9 only, r=0.05)',
    5: 'MOMENTUM        (manual — x1 ridge 0.027→0.010)',
    6: 'RECOVERY        (return to W8 best, r=0.03)',
    7: 'MOMENTUM        (W9 best, r=0.08)',
    8: 'ULTRA-TIGHT     (W5 best, r=0.04)',
}

for i in range(1, 9):
    print(f'F{i}: {all_portals[i]}')
    print(f'     [{all_policies[i]}]')
    print()

print('=' * 70)
print()
print('All-time bests (updated after W9):')
print()
for i in range(1, 9):
    best_val  = all_time_best[i][0]
    best_week = all_time_best[i][1]
    print(f'  F{i} ({descriptions[i]}):  {best_val:.4f}  ({best_week})')

In [ ]:
weekly_results_all={
    1:[new_y_w1[i] for i in range(1,9)],2:[new_y_w2[i] for i in range(1,9)],
    3:[new_y_w3[i] for i in range(1,9)],4:[new_y_w4[i] for i in range(1,9)],
    5:[new_y_w5[i] for i in range(1,9)],6:[new_y_w6[i] for i in range(1,9)],
    7:[new_y_w7[i] for i in range(1,9)],8:[new_y_w8[i] for i in range(1,9)],
    9:[new_y_w9[i] for i in range(1,9)]
}
fig,axes=plt.subplots(2,4,figsize=(18,8))
fig.suptitle('Capstone W1-W9 Progress (Module 21)',fontsize=14,fontweight='bold')
for idx,fn in enumerate(range(1,9)):
    ax=axes[idx//4][idx%4]; weeks=list(range(1,10))
    vals=[weekly_results_all[w][fn-1] for w in weeks]
    running_best=[max(vals[:w]) for w in range(1,len(vals)+1)]
    ax.plot(weeks,vals,'o--',color='steelblue',alpha=0.7,label='Query')
    ax.plot(weeks,running_best,'s-',color='darkorange',linewidth=2,label='Best')
    best_v,best_w=all_time_best[fn]
    ax.set_title(f'F{fn}: {descriptions[fn]}\nbest={best_v:.3e} ({best_w})',fontsize=8)
    ax.set_xlabel('Week'); ax.set_ylabel('Output'); ax.legend(fontsize=7); ax.grid(True,alpha=0.3)
plt.tight_layout()
plot_path=os.path.join(PLOTS_DIR,'w9_progress_analysis.png')
plt.savefig(plot_path,dpi=150,bbox_inches='tight'); plt.close()
print(f'Plot saved: {plot_path}')